In [1]:
import sys
import os
sys.path.insert(0,os.path.abspath('..'))

from glob import glob
from tqdm import tqdm
from onnx2torch import convert
import matplotlib.pyplot as plt
from src.utilities.get_device import Device
from src.utilities.load_dataset import load_dataset
from skl2onnx.helpers.onnx_helper import load_onnx_model
from src.utilities.evaluate_model import evaluate_validation
from src.utilities.dataloader_generator import generate_dataloader

In [2]:
dataloaders_dict = dict(
    train=generate_dataloader(load_dataset(), batch_size=32),
    validation=generate_dataloader(load_dataset(
        folder="test"), batch_size=64)
)
device = Device().get_device()

In [3]:
directory = "../_results_with_finetune/evaluation_before_finetune"
if not os.path.exists(directory):
    os.makedirs(directory)

In [4]:
models_location = sorted(glob('../_results_with_finetune/1714292227_result_BS_32_MD_16_T_0_TT_0.5_K_5/*.onnx'))
max_accuracy = 0
model_max_accuary = ""
for i, model_location in tqdm(enumerate(models_location), position=0, leave=True):
    model_name = model_location.split("/")[-1]
    onnx_model = load_onnx_model(model_location)
    torch_model = convert(onnx_model)
    accuracy = evaluate_validation(torch_model, dataloaders_dict['validation'])
    accuarcy = accuracy.item()
    with open(f"{directory}/{model_name}.txt", "w") as f:
        f.write(str(accuracy))
    if accuracy > max_accuracy:
        max_accuracy = accuarcy
        model_max_accuary = model_name
print(max_accuracy, model_max_accuary)

276it [25:18,  5.50s/it]

0.9091611905682256 net115.onnx
